In [1]:
!pip install requests numpy geopy ipywidgets

In [2]:
import requests
import numpy as np
import time
from typing import List, Dict, Any, Optional, Tuple, Union
from dataclasses import dataclass, field
import json
from IPython.display import display, HTML, clear_output, Javascript
import ipywidgets as widgets

# Geopy for geocoding
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

print("✅ All libraries imported successfully!")


✅ All libraries imported successfully!


In [3]:
class Geocoder:
    """Handles address to coordinates conversion and vice versa"""
    
    def __init__(self, user_agent: str = "jupyter_places_search"):
        self.geolocator = Nominatim(user_agent=user_agent)
        
        # Rate limited geocoding to respect API limits
        self._geocode = RateLimiter(
            self.geolocator.geocode,
            min_delay_seconds=1.0,
            error_wait_seconds=2.0,
            max_retries=2,
            swallow_exceptions=False
        )

        self._reverse_geocode = RateLimiter(
            self.geolocator.reverse,
            min_delay_seconds=1.0,
            error_wait_seconds=2.0,
            max_retries=2,
            swallow_exceptions=False
        ) 

    def geocode(self, query: str):
        """Convert address to coordinates and address details"""
        try:
            location = self._geocode(query, exactly_one=True, addressdetails=True)
            if not location:
                return None
                        
            return {
                'lat': location.latitude,
                'lon': location.longitude,
                'display_name': location.address,
            }
            
        except Exception as e:
            print(f'❌ Geocoding error: {e}')
            return None
    def reverse_geocode(self, lat: float, lon: float):
        """Convert coordinates to address details"""
        try:
            location = self._reverse_geocode((lat, lon), exactly_one=True, addressdetails=True)
            if not location:
                return None

            address = location.raw['address'] if hasattr(location, 'raw') and 'address' in location.raw else {}
                
            return {
                'display_name': location.address,
                'address': address
            }
            
        except Exception as e:
            print(f'❌ Reverse geocoding error: {e}')
            return None

print("✅ Geocoder class defined")
    

✅ Geocoder class defined


In [4]:
class LocationManager:
    """Handles different types of location inputs"""
    
    def __init__(self):
        self.geocoder = Geocoder()
        
    def process_location_input(self, location_input: Union[str, Tuple[float, float], Dict]) -> Optional[Dict]:
        """Process various location input formats"""
        
        # Handle coordinate tuples/lists
        if isinstance(location_input, (tuple, list)) and len(location_input) == 2:
            lat, lon = map(float, location_input)
            return self._create_location_result(lat, lon, "manual_coordinates")
            
        # Handle coordinate dictionaries
        elif isinstance(location_input, dict) and 'lat' in location_input and 'lon' in location_input:
            lat, lon = float(location_input['lat']), float(location_input['lon'])
            return self._create_location_result(lat, lon, "dict_coordinates")
            
        # Handle string inputs (addresses)
        elif isinstance(location_input, str):
            return self._geocode_address(location_input)
                
        else:
            raise ValueError(f"Unsupported location input format: {type(location_input)}")
    
    def _create_location_result(self, lat: float, lon: float, source: str) -> Dict:
        """Create standardized location result"""
        try:
            reverse_result = self.geocoder.reverse_geocode(lat, lon)
            display_name = reverse_result['display_name'] if reverse_result else f"Location ({lat:.4f}, {lon:.4f})"
            address_details = reverse_result.get('address', {}) if reverse_result else {}
            
            return {
                'lat': lat,
                'lon': lon,
                'display_name': display_name,
                'address_details': address_details,
                'source': source,
                'accuracy': 'high' if source.endswith('coordinates') else 'medium'
            }
        except Exception as e:
            print(f"⚠️ Error creating location result: {e}")
            return {
                'lat': lat,
                'lon': lon,
                'display_name': f"Location ({lat:.4f}, {lon:.4f})",
                'address_details': {},
                'source': source,
                'accuracy': 'unknown'
            }
    
    def _geocode_address(self, address: str) -> Optional[Dict]:
        """Geocode an address string"""
        try:
            result = self.geocoder.geocode(address)
            if result:
                return {
                    'lat': result['lat'],
                    'lon': result['lon'],
                    'display_name': result['display_name'],
                    'source': 'geocoded_address',
                    'accuracy': 'medium',
                    'original_query': address
                }
            else:
                print(f"❌ Could not geocode address: {address}")
                return None
        except Exception as e:
            print(f"❌ Geocoding failed: {e}")
            return None

print("✅ LocationManager class defined")

✅ LocationManager class defined


In [5]:
def display_places_table(places: List[Dict], max_places: int = 10):
    """Display places in a nice HTML table"""
    if not places:
        print("No places to display")
        return
        
    places_to_show = places[:max_places]
    
    html = """
    <style>
    .places-table {
        border-collapse: collapse;
        width: 100%;
        font-family: Arial, sans-serif;
    }
    .places-table th, .places-table td {
        border: 1px solid #ddd;
        padding: 8px;
        text-align: left;
    }
    .places-table th {
        background-color: #f2f2f2;
        font-weight: bold;
    }
    .places-table tr:nth-child(even) {
        background-color: #f9f9f9;
    }
    .cuisine-tag {
        background-color: #e1f5fe;
        padding: 2px 6px;
        border-radius: 3px;
        font-size: 0.9em;
    }
    </style>
    
    <table class="places-table">
    <tr>
        <th>#</th>
        <th>Name</th>
        <th>Type</th>
        <th>Cuisine</th>
        <th>Address</th>
        <th>Coordinates</th>
    </tr>
    """
    
    for i, place in enumerate(places_to_show, 1):
        name = place.get('name', 'Unknown')
        amenity = place.get('type', {}).get('amenity', 'restaurant')
        cuisine = place.get('type', {}).get('cuisine', '-')
        
        addr = place.get('address', {})
        address_parts = []
        if 'addr:housenumber' in addr:
            address_parts.append(addr['addr:housenumber'])
        if 'addr:street' in addr:
            address_parts.append(addr['addr:street'])
        if 'addr:city' in addr:
            address_parts.append(addr['addr:city'])
        address_str = ', '.join(address_parts) if address_parts else '-'
        
        lat, lon = place.get('lat'), place.get('lon')
        coords = f"{lat:.4f}, {lon:.4f}" if lat and lon else '-'
        
        cuisine_html = f'<span class="cuisine-tag">{cuisine}</span>' if cuisine != '-' else '-'
        
        html += f"""
        <tr>
            <td>{i}</td>
            <td><strong>{name}</strong></td>
            <td>{amenity.title()}</td>
            <td>{cuisine_html}</td>
            <td>{address_str}</td>
            <td><small>{coords}</small></td>
        </tr>
        """
    
    html += "</table>"
    
    if len(places) > max_places:
        html += f"<p><em>Showing {max_places} of {len(places)} places total</em></p>"
    
    display(HTML(html))

def quick_search(location, radius=1000, place_types=None, limit=20):
    """Quick search function"""
    client = PlacesAPIClient()
    
    if place_types is None:
        place_types = ["amenity=restaurant", "amenity=cafe"]
    
    print(f"🔍 Searching for places near: {location}")
    print("-" * 50)
    
    places = client.search_nearby(
        location=location,
        radius=radius,
        places_type=place_types,
        limit=limit
    )
    
    if places:
        print(f"\n📋 Results ({len(places)} places found):")
        display_places_table(places, max_places=15)
        return places
    else:
        print("❌ No places found")
        return []

In [6]:

class PlacesAPIClient:
    """Main client for searching nearby places using OpenStreetMap data"""
    
    def __init__(self, overpass_endpoint: str = 'https://overpass-api.de/api/interpreter'):
        self.overpass_endpoint = overpass_endpoint
        self.location_manager = LocationManager()

    def _build_overpass_query(self, lat: float, lon: float, radius: int, places_type: List[str], limit: int) -> str:
        """Constructs Overpass QL query string"""
        if not places_type:
            places_type = ['amenity=restaurant', 'amenity=cafe']
        
        queries = []
        for place in places_type:
            if '=' in place:
                k, v = place.split('=', 1)
                tag_filter = f'[\"{k}\"=\"{v}\"]'
            else:
                raise ValueError('Place type should be: amenity=restaurant or amenity=cafe')
            
            queries.append(f"""
            node(around:{radius},{lat},{lon}){tag_filter};
            way(around:{radius},{lat},{lon}){tag_filter};
            relation(around:{radius},{lat},{lon}){tag_filter};
            """)
        
        query_parts = [
            "[out:json][timeout:25];",
            "(",
            *queries,
            ");",
            f"out body {limit};",
            ">;",
            "out skel qt;"
        ]
        
        return "\n".join(query_parts)

    def search_nearby(self, location: Union[str, Tuple[float, float], Dict], 
                     radius: int = 1000, places_type: List[str] = None, 
                     limit: int = 50, verbose: bool = True) -> List[Dict]:
        """Search for nearby places"""
        location_result = self.location_manager.process_location_input(location)
        
        if not location_result:
            raise ValueError(f"Could not process location: {location}")
            
        lat, lon = float(location_result['lat']), float(location_result['lon'])
        
        if verbose:
            print(f"🗺️  Searching near: {location_result['display_name']}")
            print(f"📍 Coordinates: {lat:.6f}, {lon:.6f}")
            print(f"🔍 Radius: {radius}m | Types: {places_type or ['restaurants', 'cafes']}")
        
        query = self._build_overpass_query(lat, lon, radius, places_type or [], limit)
        
        try:
            response = requests.post(
                self.overpass_endpoint,
                data=query,
                headers={'Content-Type': 'text/plain'},
                timeout=30
            )
            response.raise_for_status()
            data = response.json()
            
            if 'elements' not in data:
                if verbose:
                    print("⚠️ No elements found in response")
                return []
                
            places = self._process_elements(data['elements'])
            
            for place in places:
                place['search_location'] = location_result
                place['search_radius'] = radius
            
            if verbose:
                print(f"✅ Found {len(places)} places")
            
            return places
            
        except Exception as e:
            print(f'❌ Error with Overpass API: {e}')
            return []

    def _process_elements(self, elements: List[Dict]) -> List[Dict]:
        """Process raw OSM elements into structured place data"""
        places = []
        
        for element in elements:
            if 'tags' not in element or 'name' not in element['tags']:
                continue

            place = {
                'id': f"{element['type']}_{element['id']}",
                'name': element['tags']['name'],
                'type': self._get_place_type(element['tags']),
                'tags': element['tags'],
                'lat': element.get('lat'),
                'lon': element.get('lon')
            }

            if 'center' in element:
                place.update({
                    'lat': element['center'].get('lat'),
                    'lon': element['center'].get('lon')
                })

            address_info = {}
            tags = element.get('tags', {})
            address_keys = ['addr:street', 'addr:housenumber', 'addr:postcode', 'addr:city']
            
            for key in address_keys:
                if key in tags:
                    address_info[key] = tags[key]
                    
            if address_info:
                place['address'] = address_info
                
            places.append(place)

        return places
    
    def _get_place_type(self, tags: Dict[str, str]) -> Dict[str, str]:
        """Extract key features from OSM tags"""
        important_tags = ['cuisine', 'amenity', 'website', 'phone', 
                         'indoor_seating', 'outdoor_seating', 'wheelchair', 'takeaway']
        features = {}
        
        for tag in important_tags:
            if tag in tags:
                features[tag] = tags[tag]
                
        return features

print("✅ PlacesAPIClient class defined")


✅ PlacesAPIClient class defined


In [7]:
# First declare our global variables
global search_near_me_coords, search_near_me_params
search_near_me_coords = None
search_near_me_params = {'radius': 1000, 'place_types': ["amenity=restaurant", "amenity=cafe"], 'limit': 15}

def get_current_location():
    """Get user's current location with working button"""
    out = widgets.Output()
    display(out)
    
    js = """
    // First make sure we have the output element
    var output_area = this;
    
    new Promise(function(resolve, reject) {
        navigator.geolocation.getCurrentPosition(
            function(position) {
                resolve({
                    'lat': position.coords.latitude,
                    'lon': position.coords.longitude,
                    'accuracy': position.coords.accuracy
                });
            },
            function(error) {
                reject(error.message || 'Geolocation failed');
            },
            {
                enableHighAccuracy: true,
                timeout: 10000,
                maximumAge: 0
            }
        );
    }).then(function(position) {
        // Create a container for our results
        var container = document.createElement('div');
        container.style.margin = '20px';
        container.style.padding = '15px';
        container.style.border = '1px solid #ddd';
        container.style.borderRadius = '5px';
        
        // Add the location info
        container.innerHTML = `
            <h4>Location Found!</h4>
            <p>Latitude: ${position.lat.toFixed(6)}</p>
            <p>Longitude: ${position.lon.toFixed(6)}</p>
            <button style="
                background-color: #4CAF50;
                border: none;
                color: white;
                padding: 10px 20px;
                text-align: center;
                text-decoration: none;
                display: inline-block;
                font-size: 16px;
                margin: 4px 2px;
                cursor: pointer;
                border-radius: 5px;
            " onclick="
                var kernel = IPython.notebook.kernel;
                kernel.execute('search_near_me_coords(' + position.lat + ',' + position.lon + ')');
            ">
                🔍 Search Near Me
            </button>
        `;
        
        // Append to the output area
        output_area.append(container);
    }).catch(function(error) {
        var errorDiv = document.createElement('div');
        errorDiv.style.color = 'red';
        errorDiv.innerHTML = '<p>Error: ' + error + '</p>';
        output_area.append(errorDiv);
    });
    """
    
    display(Javascript(js))
    
    # Define our callback function
    def search_near_me_coords(lat, lon):
        with out:
            clear_output()
            print("📍 Searching near your current location...")
            client = PlacesAPIClient()
            places = client.search_nearby(
                location=(float(lat), float(lon)),
                radius=search_near_me_params['radius'],
                places_type=search_near_me_params['place_types'],
                limit=search_near_me_params['limit'],
                verbose=True
            )
            if places:
                display_places_table(places)
            else:
                print("No places found near your location")
    
    # Make it available globally
    globals()['search_near_me_coords'] = search_near_me_coords

def search_near_me(radius=1000, place_types=None, limit=15):
    """One-click location search with working button"""
    if place_types is None:
        place_types = ["amenity=restaurant", "amenity=cafe"]
    
    # Update parameters
    global search_near_me_params
    search_near_me_params = {
        'radius': radius,
        'place_types': place_types,
        'limit': limit
    }
    
    display(HTML("""
    <style>
    .location-btn {
        background-color: #4CAF50;
        border: none;
        color: white;
        padding: 10px 20px;
        text-align: center;
        text-decoration: none;
        display: inline-block;
        font-size: 16px;
        margin: 4px 2px;
        cursor: pointer;
        border-radius: 5px;
    }
    .location-container {
        margin: 20px;
        padding: 15px;
        border: 1px solid #ddd;
        border-radius: 5px;
        background-color: #f9f9f9;
    }
    </style>
    <div class="location-container">
        <h3>Find Places Near Me</h3>
        <p>Click below to search for restaurants and cafes near your current location</p>
        <button class="location-btn" onclick="IPython.notebook.kernel.execute('get_current_location()')">
            🌍 Use My Location
        </button>
    </div>
    """))

print("✅ Automatic location detection with working button")

✅ Automatic location detection with working button


In [8]:

print("# Search near current location:")
print('# Use the location widget below to find places near you')




# Location-based search implementation
from ipywidgets import Button, Output, VBox, HTML
import ipywidgets as widgets
from IPython.display import display, Javascript, clear_output

# Create output area for displaying results
location_output = Output()

def get_current_location(b):
    """Get current location when button is clicked"""
    with location_output:
        clear_output()
        print("🌍 Getting your location... (please allow browser permissions)")
        
        js = """
        new Promise(function(resolve, reject) {
            navigator.geolocation.getCurrentPosition(
                function(position) {
                    var lat = position.coords.latitude;
                    var lon = position.coords.longitude;
                    var kernel = IPython.notebook.kernel;
                    kernel.execute("process_location(" + lat + "," + lon + ")");
                },
                function(error) {
                    var kernel = IPython.notebook.kernel;
                    kernel.execute('location_output.append_stdout("❌ Error: " + "' + 
                                 error.message + '")');
                },
                {
                    enableHighAccuracy: true,
                    timeout: 10000,
                    maximumAge: 0
                }
            );
        });
        """
        display(Javascript(js))

def process_location(lat, lon):
    """Process the obtained location coordinates"""
    with location_output:
        clear_output()
        print(f"📍 Found your location: {lat:.6f}, {lon:.6f}")
        print("🔍 Searching for nearby places...")
        
        client = PlacesAPIClient()
        places = client.search_nearby(
            location=(lat, lon),
            radius=1500,
            places_type=["amenity=restaurant", "amenity=cafe"],
            limit=15,
            verbose=True
        )
        
        if places:
            display_places_table(places)
        else:
            print("❌ No places found near your location")

# Create the button
location_button = Button(
    description="🌍 Use My Location",
    button_style='success',
    icon='map-marker-alt',
    layout=widgets.Layout(width='200px', height='40px')
)
location_button.on_click(get_current_location)

# Create HTML widget for the header
header_html = HTML("""
<div style="margin:20px">
    <h3>Find Places Near Me</h3>
    <p>Click below to search for restaurants and cafes near your current location</p>
</div>
""")

# Display the interface using proper widgets
display(VBox([
    header_html,
    location_button,
    location_output
]))

# Make the processing function available globally
globals()['process_location'] = process_location
print("✅ Reliable location detection ready")

# Search near current location:
# Use the location widget below to find places near you


✅ Reliable location detection ready
